In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.mlab as mlab
from scipy.stats import norm
plt.style.use('fivethirtyeight')

In [ ]:
traffic = pd.read_csv('train_revised_11.09.2018/train_revised.csv')
traffic.columns

In [ ]:
t = traffic[['ride_id', 'travel_date', 'travel_time','travel_from', 'car_type', 'max_capacity']]

In [ ]:
t.info()

In [ ]:
t.dtypes

In [ ]:
t.describe(include='all')

In [ ]:
t.head()

Here I wanted to have a look at the fields that had 2 unique values. 

In [ ]:
print(t['car_type'].value_counts())
print(t['max_capacity'].value_counts())

So if you look at the describe cell above you'll find something curious with the travel times. Despite the fact that there over 6249 trips, and there's only 78 unquie times, graphed below. Looking at the values, majority of the first 10 are in the morning at 7, tells me that I would make sense to simplify this field to an hour field instead.

In [ ]:
t['travel_time'].value_counts().plot.bar()

In [ ]:
t['travel_time'].value_counts()

Creating the new variable. I'm just going to call it hour_booked

In [ ]:
t['hour_booked'] = pd.to_numeric(t['travel_time'].str.extract(r'(^\d*)').loc[:,0])

In [ ]:
t.head()

In [ ]:
t['hour_booked'].value_counts().sort_index().plot.bar()

In [ ]:
((t['hour_booked'].value_counts()/t['hour_booked'].value_counts().sum())*100).sort_index()

Wow that's most people booking tickets at 7 am. Over 55%. I'm thinking that I might change this into a categorical variable when I do my models. 

Next I think we can look at dates. We have 149 days, let's make that field more configurable

In [ ]:
t.head()

In [ ]:
t['travel_date'] = pd.to_datetime(t['travel_date'])

So first I looked at the year assuming they came from same year but that would have been a mistakes at the data is from over 2017, 2018. Also the busiest months are the end of the year and the beginning. Added variable for month/year for further exploration.

In [ ]:
t['m_y'] = t['travel_date'].dt.strftime('%Y-%m')
t['day'] = t['travel_date'].dt.strftime('%w')

In [ ]:
t['m_y'].value_counts().sort_index().plot.bar()

So it seems that for a lot of months in 2017, bookings were made on single days. That might be a problem. Given that the test data for this is for 2018. We'll have to keep an eye on whether removing those months makes sense. I'd say some of this is an articially made dataset.

In [ ]:
for x in t['m_y'].value_counts().sort_index().index:
    print('For month: {}'.format(x))
    print(t[t['m_y']==x]['day'].value_counts().shape)

In [ ]:
t.head()

Let's seperate the dataset based on the car type

In [ ]:
b = t[t['car_type']=='Bus']
s = t[t['car_type']=='shuttle']

So let's see how often buses travel without filling.

In [ ]:
(b['ride_id'].value_counts().value_counts().sort_index(ascending=False)/b['ride_id'].value_counts().value_counts().sort_index(ascending=False).sum()).plot.bar()

In [ ]:
(s['ride_id'].value_counts().value_counts().sort_index(ascending=False)/s['ride_id'].value_counts().value_counts().sort_index(ascending=False).sum()).plot.bar()

Looks like a lot of the buses are leaving with only a few passengers. This tells me that they're not only using this platform to book people. Either that or this dataset is not complete or artificially made. Anyway lemme continue exploring the hypothesis I was looking at before this happened. What times is a driver likely to leave without filling his ride.

In [ ]:
b.head()

In [ ]:
bpf = (b['ride_id'].value_counts()/49).round(2)
spf = (s['ride_id'].value_counts()/11).round(2)


In [ ]:
for x in bpf.index:
    b.loc[b['ride_id'].isin([x]), 'p_filled'] = bpf[x]

In [ ]:
b.head()

In [ ]:
for x in spf.index:
    s.loc[s['ride_id'].isin([x]), 'p_filled'] = spf[x]

In [ ]:
s.head()

In [ ]:
s['hour_booked'].value_counts().sort_index().index

In [ ]:
s[s['hour_booked']==5]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==6]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==7]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==8]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==9]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==10]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
s[s['hour_booked']==11]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
(s['p_filled'].value_counts().sort_index()/s['p_filled'].value_counts().sum()).plot.bar()

So from these plots you can see that shuttle drivers are more likely to leave the closer their rides get to filled(exponentially), this could be used to come up with a real time prediction system one day. Let's look at the larger buses

In [ ]:
n = s[['hour_booked','p_filled']]

In [ ]:
n

In [ ]:
for x in s['hour_booked'].value_counts().sort_index().index:
    print(s[s['hour_booked']==x]['p_filled'].mean())

In [ ]:
b['hour_booked'].value_counts().sort_index().index

In [ ]:
b[b['hour_booked']==5]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
b[b['hour_booked']==6]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
b[b['hour_booked']==7]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
b[b['hour_booked']==8]['p_filled'].value_counts().sort_index().plot.bar()

In [ ]:
(b[b['hour_booked']==19]['p_filled'].value_counts().sort_index()/b[b['hour_booked']==19]['p_filled'].value_counts().sum()).plot.bar()

In [ ]:
b[b['hour_booked']==23]['p_filled'].value_counts().sort_index().plot.bar()

It seems that bus drivers are less concern about filling their rides before they leave

The first time I went through this data set I spent so much time looking at the seats only to realise that the test set did not have any data on that. Also it doesn't have payment method either. Sigh, that's a whole morning gone. Always remember to look at the client brief people.